In [3]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent


# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-4o-mini")

In [4]:
from dataclasses import dataclass

@dataclass
class Context:
  user_name: str


In [7]:
from langchain.agents.middleware import before_model

@before_model
def log_before_model(state, runtime) :
    print(f"(step1)before_model_state : \n{state}")
    print(f"(step2)before_model_runtime : \n{runtime}")
    print(f"(step3)사용자 이름 : {runtime.context.user_name}")
    return None


In [8]:
agent = create_agent(
    model="gpt-5-mini",
    tools=[],
    middleware=[log_before_model],
    context_schema=Context,
)

파이썬의 **`dataclass`**는 데이터를 담는 클래스를 아주 쉽고 깔끔하게 만들 수 있게 도와주는 도구입니다. (파이썬 3.7부터 도입된 내장 라이브러리입니다.)

보통 클래스를 만들 때 반복적으로 작성해야 하는 번거로운 코드들을 자동으로 생성해 줍니다.

---

### 1. `dataclass`를 쓰지 않았을 때 (기존 방식)
데이터를 저장하는 클래스를 만들려면 아래와 같이 `__init__` 함수를 직접 써야 합니다.
```python
class Context:
    def __init__(self, user_name: str):
        self.user_name = user_name

# 사용
ctx = Context(user_name="John Doe")
print(ctx.user_name)
```

### 2. `dataclass`를 썼을 때 (현재 코드 방식)
`@dataclass` 데코레이터만 붙이면 `__init__`을 쓸 필요가 없습니다.
```python:1:6:03/01.ipynb
from dataclasses import dataclass

@dataclass
class Context:
  user_name: str
```
파이썬이 내부적으로 **"아, 이 클래스는 `user_name`이라는 데이터를 받는 `__init__` 함수가 필요하겠구나"**라고 판단해서 자동으로 만들어 줍니다.

---

### 3. `dataclass`가 자동으로 해주는 일들

1.  **`__init__` 생성:** 인스턴스를 만들 때 값을 전달받는 생성자를 자동으로 만듭니다.
2.  **`__repr__` 생성:** `print(ctx)`를 했을 때 `Context(user_name='John Doe')`처럼 보기 좋게 출력되도록 해줍니다. (일반 클래스는 메모리 주소값이 나옵니다.)
3.  **`__eq__` 생성:** 두 객체의 값이 같은지 비교(`==`)할 때, 메모리 주소가 아닌 **데이터 내용**을 기준으로 비교하게 해줍니다.

### 4. 왜 여기서 사용하나요?
LangChain 에이전트에서 **`context`**는 사용자 이름, 설정값 등 **단순한 데이터들의 묶음**인 경우가 많습니다. 
- 복잡한 로직(함수)보다는 **데이터 그 자체**를 안전하게 전달하는 것이 목적이기 때문에, 코드가 간결하고 읽기 쉬운 `dataclass`를 사용하는 것이 관례입니다.

### 요약
- **`dataclass`**: "데이터 저장용 클래스를 만들 때 귀찮은 코드를 대신 짜주는 도우미"
- **장점**: 코드가 짧아지고, 가독성이 좋아지며, 데이터 비교나 출력이 편리해집니다.

In [9]:
agent.invoke(
    {"messages": [{"role": "user", "content": "제 이름이 뭐죠?"}]},
    context=Context(user_name="Jay"),
)


(step1)before_model_state : 
{'messages': [HumanMessage(content='제 이름이 뭐죠?', additional_kwargs={}, response_metadata={}, id='2ec93f0a-0e76-43d4-86af-fe23659c5fd9')]}
(step2)before_model_runtime : 
Runtime(context=Context(user_name='Jay'), store=None, stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x113a5b1a0>, previous=None)
(step3)사용자 이름 : Jay


{'messages': [HumanMessage(content='제 이름이 뭐죠?', additional_kwargs={}, response_metadata={}, id='2ec93f0a-0e76-43d4-86af-fe23659c5fd9'),
  AIMessage(content='모르겠어요 — 이 채팅에선 아직 이름을 알려주지 않으셨어요. 이름을 알려주시면 이후 대화에서 그 이름으로 불러드릴게요. (참고: 대화 세션을 벗어나면 제가 기억하지 못할 수 있습니다.)', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 391, 'prompt_tokens': 12, 'total_tokens': 403, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DLhTz8Z0IRForcrr3J5xw8cQU1XiJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d0e7a-8ef2-7d42-9e28-787dba4939e4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 391, 'total_tokens': 403, 'inpu

In [11]:
from langchain.agents.middleware import wrap_model_call
from langchain_openai import ChatOpenAI

# 작성 예시
@wrap_model_call
def dynamic_model_selector(request, handler):
    # 최근 사용자의 입력 메시지 추출
    last_msg = request.messages[-1].content if request.messages else ""
    msg_len = len(last_msg)

    # 길이에 따라 모델 선택
    if msg_len < 10:
        model_name = "gpt-5-nano"
    elif msg_len < 30:
        model_name = "gpt-5-mini"
    else:
        model_name = "gpt-5"

    # request.model을 새로운 모델로 교체
    new_model = ChatOpenAI(model_name=model_name)
    new_request = request.override(model=new_model)

    # 수정된 요청으로 LLM 호출
    return handler(new_request)


----------

In [ ]:
from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.messages import HumanMessage, SystemMessage

@wrap_model_call
def inject_user_name(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    print(f"request : \n{request}")

    # 1. request 보따리 안의 책상(runtime)에서 신분증 정보 추출
    user_name = request.runtime.context.user_name

    if user_name:
        system_message = f"사용자의 이름은 {user_name}입니다"
        # 2. request.override()를 사용해 시스템 프롬프트 덮어쓰기 (모델에게 직접 떠먹여 주기)
        request = request.override(system_prompt=system_message)

    # 3. 조작된 request를 handler를 통해 모델로 전송
    return handler(request)


In [12]:
from typing import Callable
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.messages import HumanMessage, SystemMessage

@wrap_model_call
def inject_user_name(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    print(f"request : \n{request}")

    # 1. request 보따리 안의 책상(runtime)에서 신분증 정보 추출
    user_name = request.runtime.context.user_name

    if user_name:
        system_message = f"사용자의 이름은 {user_name}입니다"
        # 2. request.override()를 사용해 시스템 프롬프트 덮어쓰기 (모델에게 직접 떠먹여 주기)
        request = request.override(system_prompt=system_message)

    # 3. 조작된 request를 handler를 통해 모델로 전송
    return handler(request)


-----

In [13]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-mini",
    tools=[],
    middleware=[inject_user_name],
    context_schema=Context,
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "제 이름이 뭐죠?"}]},
    context=Context(user_name="Jay Pak"),
)
print(result["messages"][-1].content) 


request : 
ModelRequest(model=ChatOpenAI(profile={'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x113b04a50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x113b05310>, root_client=<openai.OpenAI object at 0x113b047d0>, root_async_client=<openai.AsyncOpenAI object at 0x113b056d0>, model_name='gpt-5-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True), messages=[HumanMessage(content='제 이름이 뭐죠?', additional_kwargs={}, response_metadata={}, id='41758c35-5dea-4

**미들웨어(Middleware)**와 **훅(Hook)**의 관계는 **"무엇을(Middleware)"** 하느냐와 **"언제(Hook)"** 하느냐의 관계라고 이해하시면 가장 쉽습니다.

질문하신 코드(@01.ipynb)를 바탕으로 두 개념의 관계를 정리해 드릴게요.

---

### 1. 미들웨어는 "기능의 단위"입니다.
`log_before_model`이라는 함수 자체가 하나의 **미들웨어**입니다. 
- 미들웨어는 에이전트의 실행 흐름 중간(Middle)에 끼어들어서 특정 작업(로그 남기기, 데이터 수정 등)을 수행하는 **소프트웨어 조각**입니다.

### 2. 훅은 "미들웨어가 실행될 지점"입니다.
`@before_model`이라는 데코레이터가 바로 **훅(Hook)**입니다.
- "이 미들웨어를 언제 실행할까?"라는 질문에 대해 "모델 실행 직전(before_model)이라는 갈고리(Hook)에 걸어줘!"라고 지정하는 것입니다.

---

### 3. 코드에서의 관계 (비유)

```python:1:8:03/01.ipynb
# 1. 훅(@before_model)에 걸려있는 미들웨어(log_before_model)를 정의함
@before_model
def log_before_model(state, runtime):
  ...

# 2. 에이전트를 만들 때 이 미들웨어를 장착함
agent = create_agent(
  ...
  middleware=[log_before_model], # 미들웨어 리스트에 추가
)
```

이 관계를 **노드 스타일 훅** 관점에서 다시 설명하면 이렇습니다.

*   **에이전트 시스템:** 거대한 공장 라인
*   **노드(Node):** 공장 라인의 각 기계 (LLM 기계, 도구 기계 등)
*   **훅(Hook):** 기계 바로 앞에 설치된 **센서 장착 지점**
*   **미들웨어(Middleware):** 그 지점에 실제로 장착한 **센서 기기**

### 4. 왜 "미들웨어"라고 부르나요?
사용자의 요청이 들어와서 최종 답변이 나가는 그 **중간(Middle)** 과정 어딘가에서 동작하기 때문입니다. 

- 만약 훅이 없다면, 우리는 에이전트의 복잡한 내부 코드를 다 뜯어고쳐야 로그를 남길 수 있을 것입니다. 
- 하지만 **훅(지점)**이 미리 정의되어 있기 때문에, 우리는 **미들웨어(기능)**만 만들어서 쓱 끼워 넣기만 하면 되는 것입니다.

### 요약
- **훅(Hook):** 실행 시점 (언제?) -> `@before_model`
- **미들웨어(Middleware):** 실행할 코드 (무엇을?) -> `log_before_model`
- **관계:** "미들웨어를 특정 훅 지점에 등록해서 사용한다."

즉, **미들웨어는 훅이라는 갈고리에 걸려있는 실제 작업 도구**라고 생각하시면 됩니다!

-------

# 🛠 LangChain Hooks & Middleware 가이드

## 1. Node-style Hook
**입력 파라미터:** `state`, `runtime`  
> **"노드 실행 중 특정 지점에서 순차적으로 동작하는 방식"**

공식 문서에서 로깅, 검증, 상태 업데이트에 이 방식을 추천하는 이유는 노드 내부의 **상세한 맥락(Context)**에 직접 접근할 수 있기 때문입니다.

*   **동작 원리:** 노드 함수가 실행되는 도중, 코드에 명시된 특정 지점(Execution Points)에서 훅이 호출됩니다.
*   **주요 특징:**
    *   **순차 실행:** 노드 로직과 함께 위에서 아래로 흐름을 같이 합니다.
    *   **내부 접근:** 노드 안의 로컬 변수나 인자값을 즉시 검사(Validation)하고 기록(Logging)하기 좋습니다.

### 📋 공식 문서 권장 용도
*   **Validation:** 입력 데이터가 비즈니스 로직에 들어가기 직전 검사.
*   **Logging:** 노드 내부에서 발생하는 상세 작업 단계 기록.
*   **State Updates:** 실행 결과에 따라 즉각적으로 상태를 변경해야 할 때.

---

## 2. Wrap-style Hook
**입력 파라미터:** `request`, `handler`  
> **"Model이나 Tool이 호출될 때 실행과 제어를 가로채서 원하는 작업을 수행"**

공식 문서에서 정의한 Wrap-style의 용도는 크게 세 가지입니다.

### ① 단락 실행 (Short-circuit / Zero times)
실제 도구나 노드를 단 한 번도 실행하지 않고 결과를 돌려주는 기능입니다.
*   **Caching (캐싱):** 이미 똑같은 질문에 대한 답이 있다면, 비싼 LLM이나 도구를 호출하지 않고 저장된 값을 즉시 반환합니다.
*   **Emulation:** 실제 API를 호출하는 대신 가짜 데이터를 반환할 때 사용합니다.

### ② 정상 흐름 (Normal flow / Once)
일반적인 실행이지만, 입력값이나 출력값을 가공(Transformation)할 때 사용합니다.
*   **Transformation (변환):** 도구에 들어가기 전 인자값을 보안 처리하거나, 도구가 내뱉은 복잡한 JSON을 에이전트가 읽기 쉬운 텍스트로 변환합니다.

### ③ 반복 실행 (Retry logic / Multiple times)
도구가 실패했을 때, 성공할 때까지 여러 번 다시 시도하게 만드는 기능입니다.
*   **Retries (재시도):** 네트워크 오류로 API 호출이 실패하면, 노드 내부 로직과 상관없이 미들웨어 수준에서 3번 더 시도하도록 강제합니다.

> **종류:**
> * `wrap_model_call`: 각 모델 호출(Model Call) 주변에서 작동
> * `wrap_tool_call`: 각 도구 호출(Tool Call) 주변에서 작동

---

## 🔍 주요 개념 이해하기

### 1. handler 란?
`handler`는 **"실제 모델(LLM) 호출을 실행하는 함수"** 또는 **"다음 실행 단계로 넘겨주는 함수"**를 의미합니다.

`@wrap_model_call` 데코레이터가 붙은 미들웨어는 모델이 호출되기 직전에 실행을 가로챕니다. 이때 `handler`를 호출해야만 비로소 실제 LLM이 실행됩니다.

```python
@wrap_model_call
def wrap_model_call_hook(request, handler):
    # 1. 모델 호출 전 처리 (예: 프롬프트 수정)
    
    # 2. 실제 모델 호출 실행 (handler 호출 필수!)
    result = handler(request) 
    
    # 3. 모델 호출 후 처리 (예: 결과 가공)
    return result
```

### 2. request 란?
`request`는 **"모델(LLM)이나 도구(Tool)를 호출하기 위해 필요한 모든 정보를 담고 있는 데이터 꾸러미"**입니다. 여기에는 프롬프트, 파라미터, 설정값 등이 포함됩니다.